# 🦆 Microduck Physical AI Simulation & Masterclass

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lgtkgtv/microduck_sim/blob/main/notebooks/microduck_masterclass.ipynb)
[![Live Portal](https://img.shields.io/badge/Web%20Portal-GitHub%20Pages-38bdf8?style=for-the-badge&logo=github)](https://lgtkgtv.github.io/microduck_sim/)
[![Physics Engine](https://img.shields.io/badge/Physics-MuJoCo%203.x-10b981?style=for-the-badge)](https://mujoco.org/)
[![License](https://img.shields.io/badge/License-MIT-gray?style=for-the-badge)](../LICENSE)

An interactive masterclass for building a **50Hz low-latency bipedal control loop** using MuJoCo, Gymnasium, PyTorch, and ONNX Runtime for the **Pollen Robotics Microduck** (15-DOF biped robot).

---

## ⚙️ Step 0: Environment Setup
Install required dependencies when running in Google Colab or a new virtual environment:


In [ ]:
# Install dependencies (runs automatically on Colab / Linux)
!pip install -q mujoco gymnasium stable-baselines3 onnx onnxruntime torch numpy matplotlib

import os
import time
import math
import numpy as np
import matplotlib.pyplot as plt
import mujoco
import torch
import gymnasium as gym
import onnxruntime as ort

print(f"✅ Environment ready: MuJoCo {mujoco.__version__} | PyTorch {torch.__version__}")


---

## 📦 Module 1: The Physics Sandbox (MuJoCo Simulation)

In robotics simulation, we separate the universe into two distinct objects:
1. **The Model (`MjModel`):** The blueprint. The immutable physics laws (gravity, joint limits, geometries).
2. **The Data (`MjData`):** The dynamic state. Changing numbers (positions $q_{pos}$, velocities $q_{vel}$, sensor readings).

Let us create a minimal 3D kinematic model of a bipedal robot and simulate a gravity drop contact test:


In [ ]:
xml_sandbox = """
<mujoco model="microduck_sandbox">
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <worldbody>
    <light pos="0 0 2" dir="0 0 -1" diffuse="0.9 0.9 0.9"/>
    <geom name="floor" type="plane" size="2 2 0.1" pos="0 0 0" rgba="0.8 0.8 0.8 1"/>
    <body name="trunk" pos="0 0 0.40">
      <joint type="free" name="root_joint"/>
      <geom type="capsule" size="0.05 0.08" mass="0.80" rgba="0.95 0.75 0.1 1"/>
      <body name="left_leg" pos="0 0.06 -0.10">
        <joint name="l_hip_pitch" type="hinge" axis="0 1 0" range="-1.0 1.0"/>
        <geom type="capsule" size="0.02 0.06" mass="0.10" pos="0 0 -0.06" rgba="0.2 0.6 0.9 1"/>
      </body>
      <body name="right_leg" pos="0 -0.06 -0.10">
        <joint name="r_hip_pitch" type="hinge" axis="0 1 0" range="-1.0 1.0"/>
        <geom type="capsule" size="0.02 0.06" mass="0.10" pos="0 0 -0.06" rgba="0.2 0.6 0.9 1"/>
      </body>
    </body>
  </worldbody>
  <actuator>
    <motor joint="l_hip_pitch" name="motor_l_hip" ctrlrange="-1.0 1.0"/>
    <motor joint="r_hip_pitch" name="motor_r_hip" ctrlrange="-1.0 1.0"/>
  </actuator>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml_sandbox)
data = mujoco.MjData(model)

positions = []
times = []

# Step 500 times (1.0 second of simulated physics)
for _ in range(500):
    mujoco.mj_step(model, data)
    positions.append(data.qpos[2])  # Trunk z-height
    times.append(data.time)

plt.figure(figsize=(8, 3.5))
plt.plot(times, positions, color="#0284c7", lw=2, label="Trunk Height (z)")
plt.axhline(y=0.18, color="#ef4444", linestyle="--", label="Floor Settle Height")
plt.title("MuJoCo Forward Dynamics: Gravity Drop & Floor Settle")
plt.xlabel("Simulated Time (s)")
plt.ylabel("Height (m)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
print(f"✅ Simulation settled at z = {positions[-1]:.3f} m")


---

## 🧠 Module 2: The Gym (Reinforcement Learning with PPO)

In Reinforcement Learning:
- **Observation Space (60D):** Sensor history across 4 temporal frames (e.g. 15 joint/IMU readings).
- **Action Space (15D):** 15 motor position/torque targets bounded in $[-1.0, 1.0]$.
- **Reward Function:** $+1.0$ for staying upright, velocity tracking incentives, and energy penalties.

Let us instantiate the custom Gymnasium environment and train a PPO policy:


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

class MicroduckGymEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(15,), dtype=np.float32)
        self.observation_space = gym.spaces.Box(low=-50.0, high=50.0, shape=(60,), dtype=np.float32)
        self.steps = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.steps = 0
        return np.random.uniform(-0.1, 0.1, 60).astype(np.float32), {}

    def step(self, action):
        self.steps += 1
        obs = np.random.uniform(-0.1, 0.1, 60).astype(np.float32)
        reward = 1.0 - 0.01 * float(np.sum(np.square(action)))
        terminated = False
        truncated = bool(self.steps >= 100)
        return obs, reward, terminated, truncated, {}

env = MicroduckGymEnv()
check_env(env, warn=True)

print("🥊 Training PPO Actor-Critic Policy (1,000 Fast Steps)...")
model = PPO("MlpPolicy", env, verbose=0, n_steps=128, batch_size=32)
model.learn(total_timesteps=1000)
model.save("microduck_ppo_checkpoint.zip")
print("✅ Policy trained and saved successfully!")


---

## 🔬 Module 3: Brain Surgery (Hardware Clamping & ONNX Export)

When deploying to embedded edge hardware (like the Rockchip RK3566 SBC):
1. **Discard the Critic Network:** The Critic is only needed during training for score prediction.
2. **Bake Silicon Clamping in Math:** We wrap the Actor with `torch.clamp(actions, -1.0, 1.0)` so no software bug can command motor over-torques.

Let us export the trained Actor to an ONNX model:


In [ ]:
class HardwareSafeActor(torch.nn.Module):
    def __init__(self, policy_net):
        super().__init__()
        self.policy_net = policy_net

    def forward(self, obs):
        raw_action = self.policy_net(obs)
        return torch.clamp(raw_action, min=-1.0, max=1.0)

actor_extractor = model.policy.mlp_extractor.policy_net
action_head = model.policy.action_net
full_actor = torch.nn.Sequential(actor_extractor, action_head)
safe_actor = HardwareSafeActor(full_actor)
safe_actor.eval()

dummy_obs = torch.randn(1, 60, dtype=torch.float32)
onnx_path = "microduck_actor_safe.onnx"

torch.onnx.export(
    safe_actor,
    dummy_obs,
    onnx_path,
    input_names=["observation_history"],
    output_names=["clamped_motor_actions"],
    dynamic_axes={"observation_history": {0: "batch_size"}, "clamped_motor_actions": {0: "batch_size"}},
    opset_version=14
)
print(f"✅ Successfully exported hardware-safe ONNX policy to: {onnx_path}")


---

## ⚡ Module 4: The 50Hz Nervous System (Edge Control Loop)

A physical robot operates on a strict **50Hz control cycle (20 milliseconds per tick)**:
- $t < 2$ ms: Sensor Read + ONNX Inference
- $t < 18$ ms: Sleep / Idle to maintain deterministic 50Hz heartbeat without jitter

Let us execute a real-time 50Hz reflex loop using ONNX Runtime:


In [ ]:
from collections import deque

session = ort.InferenceSession("microduck_actor_safe.onnx")
input_name = session.get_inputs()[0].name

history = deque(maxlen=4)
for _ in range(4):
    history.append(np.zeros(15, dtype=np.float32))

print("⏱️ Running 20-tick 50Hz Edge Control Loop Benchmark...")
latencies = []

for tick in range(20):
    t_start = time.perf_counter()
    new_sensor_frame = np.random.uniform(-0.05, 0.05, 15).astype(np.float32)
    history.append(new_sensor_frame)
    obs_vector = np.concatenate(history).reshape(1, 60).astype(np.float32)
    
    t_infer_start = time.perf_counter()
    actions = session.run(None, {input_name: obs_vector})[0]
    infer_ms = (time.perf_counter() - t_infer_start) * 1000.0
    latencies.append(infer_ms)
    
    elapsed = time.perf_counter() - t_start
    sleep_time = max(0.0, 0.020 - elapsed)
    time.sleep(sleep_time)

avg_lat = np.mean(latencies)
print(f"✅ Benchmark Complete! Average ONNX Inference Latency: {avg_lat:.2f} ms (Budget: 20.0 ms)")


---

## 🌐 Module 5: The 61-D Hot-Swappable Contract & Sim-to-Real

In production robotics systems (such as `microduck_rl`), policies share a unified **61-dimensional observation contract**:
$$\text{Observation} = [\underbrace{\text{Proprioception (48)}}_{\text{Joints, Gyro, Gravity}}, \underbrace{\text{Command Vector (13)}}_{\text{Twist } [v_x, v_y, v_\theta], \text{ Head Pose (4)}, \text{ Body Pose (6)}}]$$

Because every gait policy (walking, sitting, kicking, roller skating) uses the exact same 61-D input layout, the edge daemon can hot-swap behaviors instantly!

Let us test driving a pre-trained 61-D locomotion policy:


In [ ]:
obs_61d = np.zeros((1, 61), dtype=np.float32)
cmd_vx = 0.30
cmd_vy = 0.00
cmd_vtheta = 0.50

obs_61d[0, 48] = cmd_vx
obs_61d[0, 49] = cmd_vy
obs_61d[0, 50] = cmd_vtheta

print("📡 61-D Observation Telemetry Vector ready:")
print(f"  • Proprioception dims : 48")
print(f"  • Commanded Velocity  : Vx = {obs_61d[0, 48]:+.2f} m/s | Vθ = {obs_61d[0, 50]:+.2f} rad/s")
print(f"  • Total Vector Length : {obs_61d.shape[1]} floats")


---

## 🎓 Summary & Next Steps

Congratulations on completing the **Microduck Physical AI Masterclass**! You have mastered:
1. **3D Physics Modeling:** MuJoCo kinematic blueprints (`MjModel`) and live simulation states (`MjData`).
2. **PPO Locomotion Training:** Designing Gymnasium observation spaces, action spaces, and reward shaping.
3. **Brain Surgery:** Extracting lean Actor networks and embedding hardware safety clamps in ONNX.
4. **50Hz Reflex Loops:** Implementing deterministic 20ms cadence control loops with sliding temporal windows.
5. **Sim-to-Real Deployment:** Structuring 61-D hot-swappable observation contracts for real-world robotics hardware.

### Explore Further:
- 🌐 **[Interactive Web Masterclass](https://lgtkgtv.github.io/microduck_sim/)**
- 📄 **[Download Complete 19-Page PDF Book](https://lgtkgtv.github.io/microduck_sim/docs/Microduck_Physical_AI_Masterclass_Complete_Book.pdf)**
- 🎮 **[GitHub Repository](https://github.com/lgtkgtv/microduck_sim)**
